In [ ]:
import pandas as pd
import numpy as np
import re
import string
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


In [ ]:
columns = ["id", "title", "genre", "description"]

train_df = pd.read_csv(
    "train_data.txt",
    sep=" ::: ",
    names=columns,
    engine="python",
    encoding="utf-8"
)

print("Dataset Shape:", train_df.shape)


Dataset Shape: (54214, 4)


In [ ]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"\d+", "", text)

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    text = re.sub(r"\s+", " ", text)

    return text.strip()

train_df["description"] = (
    train_df["description"]
    .fillna("")
    .apply(clean_text)
)


X = train_df["description"]

y = train_df["genre"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples :", len(X_train))
print("Testing Samples  :", len(X_test))

Training Samples : 43371
Testing Samples  : 10843


In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)


In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),

    "Logistic Regression":
        LogisticRegression(
            max_iter=2000,
            n_jobs=-1
        ),

    "Linear SVM":
        LinearSVC()
}

results = {}

best_accuracy = 0
best_model = None
best_name = None
for name, classifier in models.items():

    print("\n")
    print("Training:", name)

    pipeline = Pipeline([
        ("tfidf", tfidf),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    results[name] = accuracy

    print(f"Accuracy : {accuracy:.4f}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = pipeline
        best_name = name

print("\n")
print("BEST MODEL :", best_name)
print("ACCURACY   :", best_accuracy)

final_predictions = best_model.predict(X_test)

print("\nClassification Report\n")

print(
    classification_report(
        y_test,
        final_predictions
    )
)



Training: Naive Bayes
Accuracy : 0.4850


Training: Logistic Regression
Accuracy : 0.5819


Training: Linear SVM
Accuracy : 0.5725


BEST MODEL : Logistic Regression
ACCURACY   : 0.5819422669003044

Classification Report

              precision    recall  f1-score   support

      action       0.58      0.23      0.33       263
       adult       0.74      0.26      0.39       118
   adventure       0.71      0.11      0.19       155
   animation       0.75      0.03      0.06       100
   biography       0.00      0.00      0.00        53
      comedy       0.52      0.59      0.55      1490
       crime       0.25      0.01      0.02       101
 documentary       0.65      0.87      0.75      2619
       drama       0.53      0.80      0.64      2723
      family       0.72      0.08      0.15       157
     fantasy       0.00      0.00      0.00        65
   game-show       1.00      0.33      0.50        39
     history       0.00      0.00      0.00        49
      horror       

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
cm = confusion_matrix(
    y_test,
    final_predictions
)

print("\nConfusion Matrix Shape:", cm.shape)


Confusion Matrix Shape: (27, 27)


In [ ]:
train_df.tail()

,id,title,genre,description
41705,41706,Foe (2008),short,paul is about to face a life changing event he...
41706,41707,"""Mary: Mrs. A. Lincoln"" (2018)",drama,mary todd lincoln is one of historys most misu...
41707,41708,George Best: The Legacy (2009),documentary,featuring neverbeforeseen full and frank inter...
41708,41709,Franchesca Page (1998),comedy,the story follows the adventures of rita page ...
41709,41710,Hasta que el matrimonio nos separe (1977),comedy,santander spain before the law of divorce a yo...


In [ ]:
# thriller->murder,investigation,killer
#Sci-Fi->alien,Mars,civilization,space
#Action

#Hasta que el matrimonio nos separe -->Used This Comedy

#Test The Model

In [ ]:
sample_text = """
A detective investigates a series
of brutal murders in a dark city."""

predicted_genre = best_model.predict(
    [clean_text(sample_text)]
)

print("\nPredicted Genre:")
print(predicted_genre[0])


Predicted Genre:
thriller


#Used Model Save

In [ ]:
#joblib.dump(
 #   best_model,
  #  "best_movie_genre_model.pkl"
#)

#print("\nModel Saved Successfully")